<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICS40125 - Laboratorio N°04


**Objetivo**: Aplicar técnicas intermedias y avanzadas de análisis de datos con pandas utilizando un caso real: el Índice de Libertad de Prensa. Este laboratorio incluye operaciones de limpieza, transformación, combinación de datos, y análisis exploratorio usando `merge`, `groupby`, `concat` y otras funciones fundamentales.




**Descripción del Dataset**

El presente conjunto de datos está orientado al análisis del **Índice de Libertad de Prensa**, una métrica internacional que evalúa el nivel de libertad del que gozan periodistas y medios de comunicación en distintos países. Este índice es recopilado anualmente por la organización **Reporteros sin Fronteras**.

La base de datos contempla observaciones por país y año, e incluye tanto el valor del índice como el ranking correspondiente. A menor puntaje en el índice, mayor nivel de libertad de prensa.

**Diccionario de variables**

| Variable     | Clase    | Descripción                                                                          |
| ------------ | -------- | ------------------------------------------------------------------------------------ |
| `codigo_iso` | carácter | Código ISO 3166-1 alfa-3 que representa a cada país.                                 |
| `pais`       | carácter | Nombre oficial del país.                                                             |
| `anio`       | entero   | Año en que se registró la medición del índice.                                       |
| `indice`     | numérico | Valor numérico del Índice de Libertad de Prensa (menor valor indica mayor libertad). |
| `ranking`    | entero   | Posición relativa del país en el ranking mundial de libertad de prensa.              |


**Fuente original y adaptación pedagógica**

* **Fuente original**: [Reporteros sin Fronteras](https://www.rsf-es.org/), recopilado y publicado a través del portal del [Banco Mundial](https://tcdata360.worldbank.org/indicators/h3f86901f?country=BRA&indicator=32416&viz=line_chart&years=2001,2019).
* **Adaptación educativa**: Los archivos han sido modificados intencionalmente para incorporar desafíos técnicos que permiten aplicar los contenidos abordados en clases, tales como limpieza de datos, normalización, detección de duplicados, y combinación de fuentes.


**Descripción de los archivos disponibles**

* **`libertad_prensa_codigo.csv`**: Contiene los pares `codigo_iso` y `pais`. Incluye intencionalmente un código ISO con dos nombres distintos de país para efectos de limpieza y validación de datos.

* **`libertad_prensa_01.csv`**: Contiene registros de los años **anteriores a 2010**. Incluye las variables `PAIS`, `ANIO`, `INDICE`, y `RANKING` con nombres de columna en **mayúsculas**.

* **`libertad_prensa_02.csv`**: Contiene registros de los años **desde 2010 en adelante**. Estructura similar al archivo anterior, con nombres de columna también en **mayúsculas**.





In [1]:
import numpy as np
import pandas as pd

# lectura de datos
archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_02.csv'
 ]
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_codigo.csv')

In [12]:
# a) Consolidar df_anio y normalizar nombres de columnas
df_01 = pd.read_csv(archivos_anio[0])
df_02 = pd.read_csv(archivos_anio[1])

# Normalizar nombres de columnas a minúscula
df_01.columns = df_01.columns.str.lower()
df_02.columns = df_02.columns.str.lower()

df_anio = pd.concat([df_01, df_02], ignore_index=True)
display(df_anio.head())

,codigo_iso,anio,indice,ranking
0,AFG,2001,35.5,59.0
1,AGO,2001,30.2,50.0
2,ALB,2001,NaN,NaN
3,AND,2001,NaN,NaN
4,ARE,2001,NaN,NaN


### Identificación y corrección de inconsistencias en `df_codigos`

In [13]:
# b) Explorar y limpiar df_codigos
print("Valores únicos por codigo_iso en df_codigos antes de limpiar:")
display(df_codigos.groupby('codigo_iso')['pais'].apply(list).reset_index())

# Identificar el codigo_iso con multiples paises
inconsistent_iso = df_codigos['codigo_iso'].value_counts()
inconsistent_iso = inconsistent_iso[inconsistent_iso > 1].index[0]

# Filtrar para mantener solo el país correcto (asumimos 'Estados Unidos de America' es el correcto para USA)
df_codigos_cleaned = df_codigos[~((df_codigos['codigo_iso'] == inconsistent_iso) & (df_codigos['pais'] != 'Estados Unidos de America'))]

print("\nValores únicos por codigo_iso en df_codigos después de limpiar:")
display(df_codigos_cleaned.groupby('codigo_iso')['pais'].apply(list).reset_index())

Valores únicos por codigo_iso en df_codigos antes de limpiar:


,codigo_iso,pais
0,AFG,[Afghanistán]
1,AGO,[Angola]
2,ALB,[Albania]
3,AND,[Andorra]
4,ARE,[Emiratos Árabes Unidos]
...,...,...
175,WSM,[Samoa]
176,YEM,[Yemen]
177,ZAF,[Sudáfrica]
178,ZMB,[Zambia]



Valores únicos por codigo_iso en df_codigos después de limpiar:


,codigo_iso,pais
0,AFG,[Afghanistán]
1,AGO,[Angola]
2,ALB,[Albania]
3,AND,[Andorra]
4,ARE,[Emiratos Árabes Unidos]
...,...,...
174,VNM,[Vietnam]
175,WSM,[Samoa]
176,YEM,[Yemen]
177,ZAF,[Sudáfrica]


### Combinación de `df_anio` y `df_codigos_cleaned` en `df`

In [14]:
# c) Combinar df_anio con df_codigos_cleaned
df = pd.merge(df_anio, df_codigos_cleaned, on='codigo_iso', how='inner')
display(df.head())

,codigo_iso,anio,indice,ranking,pais
0,AFG,2001,35.5,59.0,Afghanistán
1,AGO,2001,30.2,50.0,Angola
2,ALB,2001,NaN,NaN,Albania
3,AND,2001,NaN,NaN,Andorra
4,ARE,2001,NaN,NaN,Emiratos Árabes Unidos




### 1. Consolidación y limpieza de datos

A partir de los archivos disponibles, realice los siguientes pasos:

**a)** Cree un DataFrame llamado `df_anio` que consolide la información proveniente de los archivos **`libertad_prensa_01.csv`** y **`libertad_prensa_02.csv`**, correspondientes a distintas ventanas de tiempo. Recuerde que ambos archivos tienen nombres de columnas en mayúscula, por lo que debe normalizarlas a **minúscula** para asegurar consistencia.

**b)** Explore el archivo **`libertad_prensa_codigo.csv`** e identifique el código ISO que aparece asociado a dos nombres de país distintos. Elimine el registro que corresponda a un valor incorrecto o inconsistente, conservando solo el que considere válido.

**c)** Una vez preparados los archivos, cree un nuevo DataFrame llamado `df` que combine `df_anio` con `df_codigos`, utilizando la columna `codigo_iso` como clave. Asegúrese de realizar una unión que conserve únicamente los registros que tengan coincidencia en ambas fuentes.

> **Sugerencia**:
>
> * Para unir los archivos por filas (años), utilice la función `pd.concat([...])`.
> * Para combinar información por columnas (variables), utilice `pd.merge(...)` especificando `on='codigo_iso'`.



In [10]:
# a) Consolidar df_anio y normalizar nombres de columnas
df_01 = pd.read_csv(archivos_anio[0])
df_02 = pd.read_csv(archivos_anio[1])

# Normalizar nombres de columnas a minúscula
df_01.columns = df_01.columns.str.lower()
df_02.columns = df_02.columns.str.lower()

df_anio = pd.concat([df_01, df_02], ignore_index=True)
display(df_anio.head())

,codigo_iso,anio,indice,ranking
0,AFG,2001,35.5,59.0
1,AGO,2001,30.2,50.0
2,ALB,2001,NaN,NaN
3,AND,2001,NaN,NaN
4,ARE,2001,NaN,NaN


### Identificación y corrección de inconsistencias en `df_codigos`

In [15]:
# b) Explorar y limpiar df_codigos
print("Valores únicos por codigo_iso en df_codigos antes de limpiar:")
display(df_codigos.groupby('codigo_iso')['pais'].apply(list).reset_index())

# Identificar el codigo_iso con multiples paises
inconsistent_iso = df_codigos['codigo_iso'].value_counts()
inconsistent_iso = inconsistent_iso[inconsistent_iso > 1].index[0]

# Filtrar para mantener solo el país correcto (asumimos 'Estados Unidos de America' es el correcto para USA)
df_codigos_cleaned = df_codigos[~((df_codigos['codigo_iso'] == inconsistent_iso) & (df_codigos['pais'] != 'Estados Unidos de America'))]

print("\nValores únicos por codigo_iso en df_codigos después de limpiar:")
display(df_codigos_cleaned.groupby('codigo_iso')['pais'].apply(list).reset_index())

Valores únicos por codigo_iso en df_codigos antes de limpiar:


,codigo_iso,pais
0,AFG,[Afghanistán]
1,AGO,[Angola]
2,ALB,[Albania]
3,AND,[Andorra]
4,ARE,[Emiratos Árabes Unidos]
...,...,...
175,WSM,[Samoa]
176,YEM,[Yemen]
177,ZAF,[Sudáfrica]
178,ZMB,[Zambia]



Valores únicos por codigo_iso en df_codigos después de limpiar:


,codigo_iso,pais
0,AFG,[Afghanistán]
1,AGO,[Angola]
2,ALB,[Albania]
3,AND,[Andorra]
4,ARE,[Emiratos Árabes Unidos]
...,...,...
174,VNM,[Vietnam]
175,WSM,[Samoa]
176,YEM,[Yemen]
177,ZAF,[Sudáfrica]


### Combinación de `df_anio` y `df_codigos_cleaned` en `df`

In [11]:
# c) Combinar df_anio con df_codigos_cleaned
df = pd.merge(df_anio, df_codigos_cleaned, on='codigo_iso', how='inner')
display(df.head())

,codigo_iso,anio,indice,ranking,pais
0,AFG,2001,35.5,59.0,Afghanistán
1,AGO,2001,30.2,50.0,Angola
2,ALB,2001,NaN,NaN,Albania
3,AND,2001,NaN,NaN,Andorra
4,ARE,2001,NaN,NaN,Emiratos Árabes Unidos




### 2. Exploración inicial del conjunto de datos

Una vez que hayas consolidado el DataFrame final `df`, realiza un análisis exploratorio básico respondiendo las siguientes preguntas:

#### **Estructura del DataFrame**

* ¿Cuántas **filas (observaciones)** contiene el conjunto de datos?
* ¿Cuántas **columnas** tiene el DataFrame?
* ¿Cuáles son los **nombres de las columnas**?
* ¿Qué **tipo de datos** tiene cada columna?
* ¿Hay columnas con un tipo de dato inesperado (por ejemplo, fechas como strings)?

#### **Resumen estadístico**

* Genera un resumen estadístico del conjunto de datos con `.describe()`.
  ¿Qué observas sobre los valores de `indice` y `ranking`?
* ¿Qué valores mínimo, máximo y promedio tiene la columna `indice`?
* ¿Qué países presentan los valores extremos en `indice` y `ranking`?

#### **Datos faltantes**

* ¿Cuántos valores nulos hay en cada columna?
* ¿Qué proporción de observaciones tienen valores faltantes?
* ¿Hay columnas con más del 30% de datos faltantes?

#### **Unicidad y duplicados**

* ¿Cuántos países distintos (`pais`) hay en el DataFrame?
* ¿Cuántos años distintos (`anio`) hay representados?
* ¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?

#### **Validación cruzada de columnas**

* ¿Hay inconsistencias entre el país (`pais`) y su código (`codigo_iso`)?
  (por ejemplo, un mismo código ISO asociado a más de un país)

> **Sugerencia**: Apoya tu análisis con funciones como `.info()`, `.nunique()`, `.isnull().sum()`, `.duplicated()`, `.value_counts()`, entre otras.



    

In [27]:
# Estructura del DataFrame
print(" Estructura del DataFrame")
print(f"Filas (observaciones): {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print("Nombres de las columnas:", df.columns.tolist())
print("Tipos de datos por columna:")
display(df.info())

# Resumen estadístico
print(" Resumen estadístico")
desc = df.describe()
display(desc)
print("Observaciones sobre 'indice' y 'ranking':")
print(f"- 'indice': A menor valor, mayor libertad de prensa. Los valores van desde {desc.loc['min', 'indice']} hasta {desc.loc['max', 'indice']}. El promedio es {round(desc.loc['mean', 'indice'], 2)}.")
print(f"- 'ranking': A menor valor, mejor posici\u00F3n. Los valores van desde {desc.loc['min', 'ranking']} hasta {desc.loc['max', 'ranking']}. El promedio es {round(desc.loc['mean', 'ranking'], 2)}.")

# Pa\u00EDses con valores extremos en 'indice'
min_indice_pais = df.loc[df['indice'].idxmin()][['pais', 'indice']].values
max_indice_pais = df.loc[df['indice'].idxmax()][['pais', 'indice']].values
print(f"Pa\u00EDs con menor 'indice' (mayor libertad): {min_indice_pais[0]} ({min_indice_pais[1]}) ")
print(f"Pa\u00EDs con mayor 'indice' (menor libertad): {max_indice_pais[0]} ({max_indice_pais[1]}) ")

# Datos faltantes
print("Datos faltantes")
null_counts = df.isnull().sum()
print("Valores nulos por columna:")
display(null_counts[null_counts > 0])

missing_proportion = df.isnull().sum() / len(df) * 100
print("Proporci\u00F3n de valores faltantes por columna (%):")
display(missing_proportion[missing_proportion > 0])

print("Columnas con m\u00E1s del 30% de datos faltantes:")
columns_with_high_missing = missing_proportion[missing_proportion > 30]
display(columns_with_high_missing)

# Unicidad y duplicados
print("Unicidad y duplicados")
print(f"N\u00FAmero de pa\u00EDses distintos: {df['pais'].nunique()}")
print(f"N\u00FAmero de a\u00F1os distintos: {df['anio'].nunique()}")
print(f"Existen filas duplicadas (exactamente iguales): {df.duplicated().sum()}")

# Validaci\u00F3n cruzada de columnas
print("Validaci\u00F3n cruzada de columnas")
inconsistent_pais_iso = df.groupby('codigo_iso')['pais'].nunique()
if (inconsistent_pais_iso > 1).any():
    print("S\u00ED, existen inconsistencias entre 'pais' y 'codigo_iso':")
    display(inconsistent_pais_iso[inconsistent_pais_iso > 1])
else:
    print("No se encontraron inconsistencias entre 'pais' y 'codigo_iso'.")

 Estructura del DataFrame
Filas (observaciones): 3043
Columnas: 5
Nombres de las columnas: ['codigo_iso', 'anio', 'indice', 'ranking', 'pais']
Tipos de datos por columna:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3043 entries, 0 to 3042
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   codigo_iso  3043 non-null   object 
 1   anio        3043 non-null   int64  
 2   indice      2648 non-null   float64
 3   ranking     2820 non-null   float64
 4   pais        3043 non-null   object 
dtypes: float64(2), int64(1), object(2)
memory usage: 119.0+ KB


None

 Resumen estadístico


,anio,indice,ranking
count,3043.000000,2648.000000,2820.000000
mean,2009.941176,206.739211,480.173404
std,5.786029,2703.631345,6494.364772
min,2001.000000,0.000000,1.000000
25%,2005.000000,15.250000,34.000000
50%,2009.000000,27.920000,70.000000
75%,2015.000000,41.000000,110.000000
max,2019.000000,64536.000000,121056.000000


Observaciones sobre 'indice' y 'ranking':
- 'indice': A menor valor, mayor libertad de prensa. Los valores van desde 0.0 hasta 64536.0. El promedio es 206.74.
- 'ranking': A menor valor, mejor posición. Los valores van desde 1.0 hasta 121056.0. El promedio es 480.17.
País con menor 'indice' (mayor libertad): Dinamarca (0.0) 
País con mayor 'indice' (menor libertad): Kosovo (64536.0) 
Datos faltantes
Valores nulos por columna:


...

Proporción de valores faltantes por columna (%):


,0
indice,12.980611
ranking,7.328294


Columnas con más del 30% de datos faltantes:


,0


Unicidad y duplicados
Número de países distintos: 178
Número de años distintos: 17
Existen filas duplicadas (exactamente iguales): 0
Validación cruzada de columnas
No se encontraron inconsistencias entre 'pais' y 'codigo_iso'.





### 3. Comparación regional: países latinoamericanos

En esta sección se busca identificar cuáles son los países de América Latina que han presentado los valores extremos del **Índice de Libertad de Prensa** en cada año observado.

> Recuerda que un menor puntaje en `indice` implica mayor libertad de prensa.

#### **Tareas:**

**a)** Utilizando un ciclo `for`, recorre cada año del conjunto de datos filtrado por países latinoamericanos, y determina para cada año:

* El país con el menor valor de `indice` (mayor libertad de prensa).
* El país con el mayor valor de `indice` (menor libertad de prensa).

**b)** Resuelve la misma tarea del punto anterior utilizando un enfoque vectorizado con `groupby`, sin usar ciclos explícitos.



#### **Lista de países latinoamericanos considerada:**

```python
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']
```

> Puedes usar esta lista para filtrar el DataFrame final por la columna `codigo_iso`.



In [22]:
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']

df_america = df[df['codigo_iso'].isin(america)].copy()

print(" a) Utilizando un ciclo for")
for anio in sorted(df_america['anio'].unique()):
    df_anual = df_america[df_america['anio'] == anio]

    # Filtrar NaN para indice
    df_anual_filtered = df_anual.dropna(subset=['indice'])

    if not df_anual_filtered.empty:
        min_indice_pais = df_anual_filtered.loc[df_anual_filtered['indice'].idxmin()]
        max_indice_pais = df_anual_filtered.loc[df_anual_filtered['indice'].idxmax()]

        print(f"\nAño: {anio}")
        print(f"  País con mayor libertad de prensa (menor índice): {min_indice_pais['pais']} ({min_indice_pais['indice']:.2f})")
        print(f"  País con menor libertad de prensa (mayor índice): {max_indice_pais['pais']} ({max_indice_pais['indice']:.2f})")
    else:
        print(f"\nAño: {anio}: No hay datos de índice disponibles para este año.")


print(" b) Utilizando un enfoque vectorizado con groupby")

# Encontrar el país con el índice mínimo para cada año
min_idx = df_america.dropna(subset=['indice']).groupby('anio')['indice'].idxmin()
min_libertad_por_anio = df_america.loc[min_idx]

# Encontrar el país con el índice máximo para cada año
max_idx = df_america.dropna(subset=['indice']).groupby('anio')['indice'].idxmax()
max_libertad_por_anio = df_america.loc[max_idx]

for anio in sorted(df_america['anio'].unique()):
    min_pais = min_libertad_por_anio[min_libertad_por_anio['anio'] == anio]
    max_pais = max_libertad_por_anio[max_libertad_por_anio['anio'] == anio]

    print(f"\nAño: {anio}")
    if not min_pais.empty:
        print(f"  País con mayor libertad de prensa (menor índice): {min_pais['pais'].iloc[0]} ({min_pais['indice'].iloc[0]:.2f})")
    else:
        print(f"  No hay datos para el país con mayor libertad de prensa en {anio}")

    if not max_pais.empty:
        print(f"  País con menor libertad de prensa (mayor índice): {max_pais['pais'].iloc[0]} ({max_pais['indice'].iloc[0]:.2f})")
    else:
        print(f"  No hay datos para el país con menor libertad de prensa en {anio}")

 a) Utilizando un ciclo for

Año: 2001
  País con mayor libertad de prensa (menor índice): Canadá (0.80)
  País con menor libertad de prensa (mayor índice): Cuba (90.30)

Año: 2002
  País con mayor libertad de prensa (menor índice): Trinidad y Tobago (1.00)
  País con menor libertad de prensa (mayor índice): Cuba (97.83)

Año: 2003
  País con mayor libertad de prensa (menor índice): Trinidad y Tobago (2.00)
  País con menor libertad de prensa (mayor índice): Argentina (35826.00)

Año: 2004
  País con mayor libertad de prensa (menor índice): Trinidad y Tobago (2.00)
  País con menor libertad de prensa (mayor índice): Cuba (87.00)

Año: 2005
  País con mayor libertad de prensa (menor índice): Bolivia (4.50)
  País con menor libertad de prensa (mayor índice): Cuba (95.00)

Año: 2006
  País con mayor libertad de prensa (menor índice): Canadá (4.88)
  País con menor libertad de prensa (mayor índice): Cuba (96.17)

Año: 2007
  País con mayor libertad de prensa (menor índice): Canadá (3.33)
 

### 4. Análisis anual del índice por país

En esta sección se busca analizar la evolución del **índice máximo** de libertad de prensa alcanzado por cada país a lo largo del tiempo.

#### **Tarea principal:**

* Construye una tabla dinámica (`pivot_table`) donde las **filas** correspondan a los países, las **columnas** a los años (`anio`) y los **valores** sean el `indice` máximo alcanzado por cada país en ese año.
* Asegúrate de reemplazar los valores nulos resultantes con `0`.

> **Hint**: Puedes utilizar el parámetro `fill_value=0` en `pd.pivot_table(...)`.



#### **Preguntas adicionales:**

**a)** ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
**b)** ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?

> (Pista: usa `.mean(axis=0)` sobre la tabla pivot)

**c)** ¿Qué país muestra mayor **variabilidad** (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?

> (Pista: aplica `.max(axis=1) - .min(axis=1)`)

**d)** ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?

**e)** ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?





In [28]:
# Tarea principal: Construir la tabla dinámica (pivot_table)
pivot_df = df.pivot_table(
    index='pais',
    columns='anio',
    values='indice',
    aggfunc='max',
    fill_value=0
)
display(pivot_df.head())

print("\n--- Preguntas adicionales ---")

anio,2001,2002,2003,2004,2005,2006,2007,2008,2009,2012,2013,2014,2015,2017,2018,2019
pais,,,,,,,,,,,,,,,,
Afghanistán,35.5,40.17,28.25,39.17,44.25,56.50,59.25,54.25,51.67,37.36,37.07,37.44,37.75,39.46,37.28,36.55
Albania,0.0,6.50,11.50,14.17,18.00,25.50,16.00,21.75,21.50,30.88,29.92,28.77,29.92,29.92,29.49,29.84
Alemania,1.5,1.33,2.00,4.00,5.50,5.75,4.50,3.50,4.25,10.24,10.23,11.47,14.80,14.97,14.39,14.60
Algeria,31.0,33.00,43.50,40.33,40.00,40.50,31.33,49.56,47.33,36.54,36.26,36.63,41.69,42.83,43.13,45.75
Andorra,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,6.82,6.82,19.87,19.87,21.03,22.21,24.63



--- Preguntas adicionales ---


In [34]:
# a) ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
max_indice_total = pivot_df.max().max()
pais_max_indice = pivot_df.stack().idxmax()[0]

min_indice_total = pivot_df[pivot_df > 0].min().min()
pais_min_indice = pivot_df[pivot_df > 0].stack().idxmin()[0]

print(f"a) País con el mayor índice en toda la tabla: {pais_max_indice} ({max_indice_total})")
print(f"   País con el menor índice (distinto de cero) en toda la tabla: {pais_min_indice} ({min_indice_total})")

a) País con el mayor índice en toda la tabla: Kosovo (64536.0)
   País con el menor índice (distinto de cero) en toda la tabla: Austria (0.5)


In [30]:
# b) ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?
average_indice_by_year = pivot_df.replace(0, np.nan).mean(axis=0) # Replace 0 with NaN for mean calculation
year_max_avg_indice = average_indice_by_year.idxmax()
max_avg_indice_value = average_indice_by_year.max()
year_min_avg_indice = average_indice_by_year.idxmin()
min_avg_indice_value = average_indice_by_year.min()

print(f"\nb) Año con el promedio de índice más alto: {year_max_avg_indice} ({max_avg_indice_value:.2f})")
print(f"   Año con el promedio de índice más bajo: {year_min_avg_indice} ({min_avg_indice_value:.2f})")


b) Año con el promedio de índice más alto: 2012 (471.42)
   Año con el promedio de índice más bajo: 2002 (25.75)


In [31]:
# c) ¿Qué país muestra mayor variabilidad (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?
pivot_df_no_zeros = pivot_df.replace(0, np.nan) # Replace 0 with NaN for variability calculation
variability = pivot_df_no_zeros.max(axis=1) - pivot_df_no_zeros.min(axis=1)
max_variability_country = variability.idxmax()
max_variability_value = variability.max()

print(f"\nc) País con mayor variabilidad en el índice: {max_variability_country} ({max_variability_value:.2f})")


c) País con mayor variabilidad en el índice: Kosovo (46098.00)


In [32]:
# d) ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?
constant_index_countries = variability[variability == 0].index.tolist()

print(f"\nd) Países con índice constante (variabilidad 0): {constant_index_countries}")
if not constant_index_countries:
    print("   No se encontraron países con índice constante en los años registrados.")


d) Países con índice constante (variabilidad 0): ['Antigua y Barbuda', 'Granada']


In [33]:
# e) ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?
countries_with_no_data = pivot_df[(pivot_df == 0).all(axis=1)].index.tolist()

print(f"\ne) Países sin ningún dato (todos los valores en 0): {countries_with_no_data}")
print("   Estos países quedaron con todos los valores en 0 porque no tenían registros de 'indice' en el DataFrame original para ningún año, o porque todos sus valores fueron nulos y reemplazados por 0 en la tabla dinámica.")


e) Países sin ningún dato (todos los valores en 0): []
   Estos países quedaron con todos los valores en 0 porque no tenían registros de 'indice' en el DataFrame original para ningún año, o porque todos sus valores fueron nulos y reemplazados por 0 en la tabla dinámica.
